#개요
Olist 데이터셋 가지고 text-to-sql FineTuning용 데이터셋 만들기.

#환경설정

In [1]:
!pip install langchain-openai langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.9 MB/s eta 0:00:00


In [2]:
import json
import re
import pandas as pd
import random
import sqlite3
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [127]:
os.environ["OPENAI_API_KEY"] = ""

In [117]:
llm = ChatOpenAI(model = "gpt-5", temperature = 0.3)
llm_sql = ChatOpenAI(model = 'gpt-4o-mini', temperature=0)

목표

instruction : "DDL Statements:DDL문\n입력:한글프롬프트"

input : ""

output : SQL 문

In [5]:
with open('/content/gretelai_text_to_sql_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))

FileNotFoundError: [Errno 2] No such file or directory: '/content/gretelai_text_to_sql_data.json'

데이터셋 로드

In [6]:
#Olist 데이터
df_customers        = pd.read_csv('olist_customers_dataset.csv')
df_geolocation      = pd.read_csv('olist_geolocation_dataset.csv')
df_order_items      = pd.read_csv('olist_order_items_dataset.csv')
df_order_payments   = pd.read_csv('olist_order_payments_dataset.csv')
df_order_reviews    = pd.read_csv('olist_order_reviews_dataset.csv')
df_orders           = pd.read_csv('olist_orders_dataset.csv')
df_products         = pd.read_csv('olist_products_dataset.csv')
df_sellers          = pd.read_csv('olist_sellers_dataset.csv')

In [ ]:
#base text-to-sql 데이터
!wget https://raw.githubusercontent.com/leejunho12316/LLaMA-Factory/main/data/text_to_sql_data.json

In [99]:
with open('text_to_sql_data.json') as f:
  base_data = json.load(f)

In [124]:
print(base_data[20].get('instruction'))

입력 텍스트: 지난 1년 동안 각 국가별로 발표된 자율 주행 연구 논문의 총 수는 무엇인가요?

DDL statements:
CREATE TABLE ResearchPapers (ID INT, Title VARCHAR(100), PublishedDate DATE, Author VARCHAR(50), Country VARCHAR(50)); INSERT INTO ResearchPapers (ID, Title, PublishedDate, Author, Country) VALUES (1, 'AD Research 1', '2022-01-15', 'A. Smith', 'USA'), (2, 'AD Research 2', '2022-03-20', 'B. Johnson', 'Canada'), (3, 'AD Research 3', '2021-12-12', 'C. Lee', 'South Korea'), (4, 'AD Research 4', '2022-05-08', 'D. Patel', 'India'), (5, 'AD Research 5', '2021-11-01', 'E. Chen', 'China');

위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다.


# 함수 모음

In [7]:
def parse_llm_output(llm_output: str, num_pairs: int) -> list[dict]:
    """
    LLM 출력을 파싱해 질문-SQL 딕셔너리를 담은 리스트로 반환

    Args:
        llm_output : LLM 출력 문자열
        num_pairs  : 파싱할 질문-SQL 쌍의 개수

    Returns:
        [{'Question': '...', 'SQL': '...'}, ...]
    """
    questions = re.findall(r'\[질문\]\s*(.*?)\s*(?=\[SQL\])', llm_output, re.DOTALL)
    sqls      = re.findall(r'\[SQL\]\s*(.*?)\s*(?=\[질문\]|$)',  llm_output, re.DOTALL)

    # 백틱 제거
    sqls = [re.sub(r'```sql|```', '', sql).strip() for sql in sqls]
    questions = [q.strip() for q in questions]

    result = []
    for i in range(min(num_pairs, len(questions), len(sqls))):
        result.append({
            'Question': questions[i],
            'SQL':      sqls[i]
        })

    return result

In [126]:
def _convert_to_sqlite(sql: str) -> str:
    """
    DB 종류에 따라 다른 SQL문을 LLM을 사용해 SQLite 문법으로 변환.
    execute_sql_on_db 에서만 사용하는 함수

    Args:
        sql : 변환할 SQL문

    Returns:
        SQLite 문법으로 변환된 SQL문
    """
    response = llm_sql.invoke(
        f"""다음 SQL문을 SQLite 문법으로 변환해줘.
반드시 SQL문만 출력하고 다른 설명은 절대 추가하지 마.
백틱이나 코드블록 없이 순수 SQL문만 출력해.

{sql}"""
    )
    return response.content.strip()


def execute_sql_on_db(parsed_results: list[dict], conn) -> list[dict]:
    """
    질문-SQL 딕셔너리를 담은 List를 받아 DB에 전체 실행해보기.
    """

    results = []

    for item in parsed_results:
        question  = item['Question']
        sql       = item['SQL'].rstrip(';')
        sql_sqlite = None

        # SQL 유형 판별
        sql_type = sql.strip().split()[0].upper()  # SELECT, UPDATE, DELETE, INSERT

        def run_sql(query):
            if sql_type == 'SELECT':
                # SELECT는 pd.read_sql_query() 사용
                return pd.read_sql_query(query, conn), 'success'
            else:
                # INSERT, UPDATE, DELETE는 cursor로 실행 후 롤백
                cursor = conn.cursor()
                cursor.execute(query)
                affected = cursor.rowcount
                conn.rollback()  # 실제 반영 안 되게 롤백
                return pd.DataFrame({'affected_rows': [affected]}), 'success'

        # 1차 시도: 원본 SQL 실행
        try:
            df_result, status = run_sql(sql)
        except Exception as e:
            df_result = None
            status    = f'error: {e}'

            # 2차 시도: LLM으로 SQLite 변환 후 재실행
            print(f"❌ 1차 실행 실패 → LLM으로 SQLite 변환 시도")
            try:
                sql_sqlite        = _convert_to_sqlite(sql).rstrip(';')
                df_result, status = run_sql(sql_sqlite)
            except Exception as e2:
                df_result = None
                status    = f'error (변환 후에도 실패): {e2}'

        results.append({
            'Question'  : question,
            'SQL'       : sql,
            'SQL_SQLite': sql_sqlite,
            'Result'    : df_result,
            'Status'    : status
        })

        print(f"Question:   {question}")
        print(f"SQL 원본:   {sql}")
        if sql_sqlite:
            print(f"SQL 변환:   {sql_sqlite}")
        print(f"Status:     {status}")
        print("\n")
        print(df_result if df_result is not None else "")
        print("\n")
        print("-" * 50)

    return results

In [108]:
#Prompt 예시 추가용 함수
def get_sample_values(df, n=3) -> str:
  """
  dataframe의 각 칼럼 별 unique한 값 중 랜덤으로 n개를 뽑은 결과 반환. Prompt 추가용.

  인수 :
    dataframe : olist 데이터프레임
    n : 랜덤으로 추출할 값 개수
  return :
    dataframe 각 칼럼에서 unique 한 값 중 랜덤으로 n개를 뽑은 결과 str
  """
  sample_text = ""
  for col in df.columns:
      uniques = df[col].dropna().unique().tolist()
      samples = random.sample(uniques, min(n, len(uniques)))  # 랜덤으로 n개 추출
      sample_text += f"{col} : {samples}\n"
  return sample_text

def get_sample_queries(base_data, n: int):
  """
  json 데이터 중 랜덤으로 n개의 query를 뽑은 결고 반환. Prompt 추가용
  """
  queries = [
    i.get('instruction').split('DDL statements:')[0].split('입력 텍스트:')[1].strip() for i in random.sample(base_data, n)
  ]
  return "\n".join(queries)

In [58]:
#최종 저장용 함수
def convert_to_gretel_format(total_result: list[dict], ddl: str) -> list[dict]:
    """
    total_results를 gretelai text-to-sql 학습 데이터 형식으로 변환

    Args:
        total_results : [{'Question': ..., 'SQL': ...}, ...] LLM 실행 결과 전체 List
        ddl           : instruction에 넣을 DDL 문자열
        ddl2          : 두 번째 DB의 DDL 문자열 (없으면 None)

    Returns:
        [{'instruction': ..., 'input': '', 'output': ...}, ...]
    """
    converted = []
    for item in total_result:
        question = item['Question']
        sql      = item['SQL']

        instruction = (
            f"입력 텍스트: {question}\n\n"
            f"DDL statements:\n{ddl}\n\n"
            f"위의 테이블 명세와 사용자의 입력 텍스트를 바탕으로 SQL 쿼리를 작성합니다."
        )

        converted.append({
            "instruction": instruction,
            "input"      : "",
            "output"     : f"쿼리 작성: {sql}"
        })

    return converted

# LLM 관련 설정

In [12]:
# SYSTEM_PROMPT = """
# #역할
# 당신은 text-to-SQL을 수행해야 합니다.
# DDL 선언문, 칼럼 설명, 칼럼별 값 예시를 참고해 사용자의 입장에서 할 수 있는 자연어 질문과 그에 대응하는 SQL문을 작성해주세요.
# 질문-SQL 쌍을 3개 생성하세요.

# #규칙
# 1. 반드시 코드 블록 없이 순수 SQL만 출력하세요.
# 2. history를 참고해 중복이 없게끔 하세요.
# 3. 아래 SELECT, INSERT, UPDATE, DELETE SQL 패턴을 고르게 사용할 수 있는 질문-sql 쌍을 만드세요.
#    [SELECT]
#    - 단순 조회 (SELECT, WHERE), 집계 (COUNT, AVG, SUM, MIN, MAX + GROUP BY), 정렬 및 제한 (ORDER BY, LIMIT)
#    - NULL 처리 (IS NULL, IS NOT NULL, COALESCE)
#    - 날짜 연산 (날짜 범위 조회, 날짜 차이 계산)
#    - 서브쿼리 (IN, EXISTS, 스칼라 서브쿼리)
#    - 조건 분기 (CASE WHEN), 중복 제거 (DISTINCT)
#    [INSERT]
#    - 단일 행 삽입, 다중 행 삽입
#    [UPDATE]
#    - 단일 컬럼 수정, 조건부 다중 컬럼 수정
#    [DELETE]
#    - 조건부 행 삭제
# 4. 칼럼별 값 예시를 활용해 구체적인 값을 포함한 질문을 생성하세요.

# #주의
# 1. 다음 리스트 같은 질문은 비현실적인 질문입니다
# - '2017-03-14 12:58:42'에 구매된 주문의 배송 예정일을 '2018-11-01'로 업데이트하고 싶습니다. : 사람은 날짜 단위를 시분초까지 쪼개서 요청하지 않습니다.

# 2. SQL 작성 시 주의
#    기간을 조회할 때 BETWEEN으로 끝 날짜를 지정하면 마지막 날이 누락됩니다.
#    기간 조회는 ">= 시작일 AND < 다음 기간 시작일" 패턴을, 하루 조회는 DATE() 함수를 사용하세요.
#    - 나쁨: WHERE col BETWEEN '2018-06-01' AND '2018-06-30'  (6월 30일 누락)
#    - 좋음: WHERE col >= '2018-06-01' AND col < '2018-07-01'
#    - 좋음: WHERE DATE(col) = '2018-06-04'

# 3. 현재 날짜/시간
# 현재 시각이나 오늘 날짜에 의존하는 질문·SQL을 만들지 마세요.
# "지금", "오늘", "최근 1년", "이번 달" 처럼 실행 시점에 따라 답이 달라지는 표현을 쓰지 마세요.
# NOW(), CURRENT_DATE, CURRENT_TIMESTAMP, DATE_ADD/SUB(NOW()...) 같은 함수도 사용 금지입니다.
# 날짜 조건은 '2018-05-01' 처럼 고정된 날짜 리터럴로만 작성하세요.

# #출력 형식
# 반드시 다음 형식을 지켜 출력해주세요.
# [질문]
# 자연어 질문
# [SQL]
# SQL문
# """

In [113]:
SYSTEM_PROMPT = """
#역할
당신은 Text-to-SQL을 수행해야합니다.
DDL 선언문, 칼럼 설명, 칼럼 값 예시, 질문 예시를 참고해 사용자가 할 법한 질문-SQL 쌍을 작성해주세요.
실제 사용자가 Text-to-SQL LLM에게 자연스럽게 물어볼 법한 질문과 그에 정확히 대응하는 SQL을 작성하세요.
질문-SQL쌍은 10개 생성하세요.

#최우선 중요 원칙
사람이 실제로 어떻게 질문할지 생각하세요. 그리고 그 질문에 정확히 대응하는 SQL 문을 작성하세요.
질문 예시를 적극적으로 참고하세요.

#규칙
1. 반드시 코드 블록 없이 순수 SQL만 출력하세요.
2. history를 참고해 중복이 없게끔 하세요.
3. 다음 리스트 같은 질문은 비현실적인 질문입니다
- '2017-03-14 12:58:42'에 구매된 주문의 배송 예정일을 '2018-11-01'로 업데이트하고 싶습니다. : 사람은 날짜 단위를 시분초까지 쪼개서 요청하지 않습니다.
4. SQL 작성시 주의
- BETWEEN : 기간을 조회할 때 BETWEEN으로 끝 날짜를 지정하면 마지막 날이 누락됩니다. 기간 조회는 ">= 시작일 AND < 다음 기간 시작일" 패턴을, 하루 조회는 DATE() 함수를 사용하세요.
   - 나쁨: WHERE col BETWEEN '2018-06-01' AND '2018-06-30'  (6월 30일 누락)
   - 좋음: WHERE col >= '2018-06-01' AND col < '2018-07-01'
   - 좋음: WHERE DATE(col) = '2018-06-04'
- 현재 날짜/시간 : 현재 시각이나 오늘 날짜에 의존하는 질문·SQL을 만들지 마세요.
"지금", "오늘", "최근 1년", "이번 달" 처럼 실행 시점에 따라 답이 달라지는 표현을 쓰지 마세요.
NOW(), CURRENT_DATE, CURRENT_TIMESTAMP, DATE_ADD/SUB(NOW()...) 같은 함수도 사용 금지입니다.
날짜 조건은 '2018-05-01' 처럼 고정된 날짜 리터럴로만 작성하세요.

#출력 형식
반드시 다음 형식을 지켜 출력해주세요.
[질문]
자연어 질문
[SQL]
SQL문
"""

In [79]:
SYSTEM_PROMPT_MULTI = """
#역할
당신은 text-to-SQL을 수행해야 합니다.
아래에 제공되는 2개에 DDL 선언문, 칼럼 설명, 칼럼별 값 예시를 참고해 사용자의 입장에서 할 수 있는 자연어 질문과 그에 대응하는 SQL문을 작성해주세요.
질문-SQL 쌍을 3개 생성하세요.

#규칙
1. 반드시 코드 블록 없이 순수 SQL만 출력하세요.
2. history를 참고해 중복이 없게끔 하세요.
3. 모든 질문-SQL 쌍은 반드시 두 테이블을 모두 사용해야 합니다.
   따라서 SQL은 SELECT 문으로 작성하며, 아래 패턴을 고르게 사용하세요.
   - 크로스 테이블 조인: 두 테이블을 JOIN (INNER, LEFT, RIGHT, FULL OUTER)
   - 집합 연산: 두 테이블의 결과를 UNION / UNION ALL / INTERSECT / EXCEPT로 결합
   - 서브쿼리: 한 테이블의 결과를 다른 테이블 조건으로 사용 (IN, EXISTS 등)
4. 칼럼별 값 예시를 활용해 구체적인 값을 포함한 질문을 생성하세요.

#주의
1. 다음 리스트 같은 질문은 비현실적인 질문입니다
- '2017-03-14 12:58:42'에 구매된 주문의 배송 예정일을 '2018-11-01'로 업데이트하고 싶습니다. : 사람은 날짜 단위를 시분초까지 쪼개서 요청하지 않습니다.

2. SQL 작성 시 주의
   기간을 조회할 때 BETWEEN으로 끝 날짜를 지정하면 마지막 날이 누락됩니다.
   기간 조회는 ">= 시작일 AND < 다음 기간 시작일" 패턴을, 하루 조회는 DATE() 함수를 사용하세요.
   - 나쁨: WHERE col BETWEEN '2018-06-01' AND '2018-06-30'  (6월 30일 누락)
   - 좋음: WHERE col >= '2018-06-01' AND col < '2018-07-01'
   - 좋음: WHERE DATE(col) = '2018-06-04'

3. 현재 날짜/시간
현재 시각이나 오늘 날짜에 의존하는 질문·SQL을 만들지 마세요.
"지금", "오늘", "최근 1년", "이번 달" 처럼 실행 시점에 따라 답이 달라지는 표현을 쓰지 마세요.
NOW(), CURRENT_DATE, CURRENT_TIMESTAMP, DATE_ADD/SUB(NOW()...) 같은 함수도 사용 금지입니다.
날짜 조건은 '2018-05-01' 처럼 고정된 날짜 리터럴로만 작성하세요.

#출력 형식
반드시 다음 형식을 지켜 출력해주세요.
[질문]
자연어 질문
[SQL]
SQL문
"""

In [88]:
HUMAN_PROMPT = """
# DDL 선언문
{DDL_Statement}

# 칼럼 설명
{column_descriptions}

# 칼럼 값 예시
{column_values}

#질문 예시
{question_examples}

# history
{history}
"""

In [114]:
def generate_sql(DDL_Statement: str, column_descriptions: str, column_values: str, question_examples: str,
                 history : list[dict], is_multi=False) -> str:
    """
    DDL문과 컬럼 설명을 입력받아 LLM으로 질문-SQL 쌍을 생성

    Args:
        DDL_Statement       : 테이블 DDL 문
        column_descriptions : 컬럼 설명

    Returns:
        LLM이 생성한 질문-SQL 쌍 문자열
    """

    system_prompt = SYSTEM_PROMPT_MULTI if is_multi else SYSTEM_PROMPT

    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_prompt),
        HumanMessagePromptTemplate.from_template(HUMAN_PROMPT),
    ])

    chain = prompt | llm

    # history 중 질문만 str로 취합
    history_str = ""
    for res in total_result:
      history_str += res.get('Question') + "\n"

    response = chain.invoke({
        "DDL_Statement"      : DDL_Statement,
        "column_descriptions": column_descriptions,
        "column_values" : column_values,
        "question_examples" : question_examples,
        "history" : history_str
    })

    print(response.usage_metadata) # 캐싱 작동 확인용
    return response.content

#단일 SQL 문 생성


각 DDL문은 다음 프롬프트를 사용해 얻기.
(Human Prompt에 DF으로부터 값 예시 랜덤으로 넣는 로직 삭제하고 DDL 문의 INSERT INTO ~ VALUES로 대체하기)

```
DB에 대한 DDL문을 다음 예시처럼 한 줄로 작성해줘.
DDL문을 제외한 그 어떤 출력도 하지마.
DDL문만 출력해줘.

#테이블명
orders

#DB
order_id	customer_id	order_status	order_purchase_timestamp	order_approved_at	order_delivered_carrier_date	order_delivered_customer_date	order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7	9ef432eb6251297304e76186b10a928d	delivered	2017-10-02 10:56:33	2017-10-02 11:07:15	2017-10-04 19:55:00	2017-10-10 21:25:13	2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451	b0830fb4747a6c6d20dea0b8c802d7ef	delivered	2018-07-24 20:41:37	2018-07-26 03:24:27	2018-07-26 14:31:00	2018-08-07 15:27:45	2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d	41ce2a54c0b03bf3443c3d931a367089	delivered	2018-08-08 08:38:49	2018-08-08 08:55:23	2018-08-08 13:50:00	2018-08-17 18:06:29	2018-09-04 00:00:00

#예시
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');
CREATE TABLE farmers_india (id INT, name VARCHAR(255), district_id INT, age INT, income INT); INSERT INTO farmers_india (id, name, district_id, age, income) VALUES (1, 'Farmer A', 1, 45, 50000); CREATE TABLE districts_india (id INT, name VARCHAR(255), state VARCHAR(255)); INSERT INTO districts_india (id, name, state) VALUES (1, 'District A', 'Maharashtra');
CREATE TABLE Armed_Forces (base_id INT, base_name VARCHAR(50), base_location VARCHAR(50), base_type VARCHAR(50)); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (1, 'Fort Bragg', 'North Carolina', 'Army'); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (2, 'Camp Pendleton', 'California', 'Marines');
```

## 1.df_orders

In [118]:
# 변수 설정 (df마다 1,2,3,4 수정해야함.)
  #1. DDL 선언문 : INSERT INTO VALUES 삭제
DDL_Statement = """
CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id));
"""
  #2. 칼럼 설명 (README 참고)
column_descriptions = """
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.
"""

#LLM 실행
total_result = []
for i in range(2):
  print(f'진행중 : {i+1}')

  #3. 컬럼 값 예시 (unique n개)
  column_values = get_sample_values(df_orders,3)

  #4. 질문 예시
  question_examples = get_sample_queries(base_data, 10)

  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        column_values = column_values,
                        question_examples = question_examples,
                        history = total_result)

  #LLM 응답 파싱 후 저장.
  parsed = parse_llm_output(result, 10)
  total_result.extend(parsed)

진행중 : 1
{'input_tokens': 1346, 'output_tokens': 3688, 'total_tokens': 5034, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 2752}}
진행중 : 2
{'input_tokens': 1699, 'output_tokens': 4756, 'total_tokens': 6455, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 3968}}


In [120]:
# 파싱된 SQL 전체 DB에 실행해보기.
conn = sqlite3.connect(':memory:') #원래는 sqlite3.connect('mydb.db')로 Disk에 저장. 지금은 :memory:로 RAM에 저장
df_orders.to_sql('orders', conn, if_exists='replace', index=False)
execute_res = execute_sql_on_db(total_result, conn)

Question:   2018년 1월에 구매된 주문은 총 몇 건인가요?
SQL 원본:   SELECT COUNT(*) AS order_count
FROM orders
WHERE order_purchase_timestamp >= '2018-01-01'
  AND order_purchase_timestamp < '2018-02-01'
Status:     success


   order_count
0         7269


--------------------------------------------------
❌ 1차 실행 실패 → LLM으로 SQLite 변환 시도
Question:   2017년 10월에 고객에게 배송 완료된 주문의 평균 배송 소요 시간(구매 시점부터 고객 수령까지, 일 단위)은 얼마인가요?
SQL 원본:   SELECT AVG(TIMESTAMPDIFF(DAY, order_purchase_timestamp, order_delivered_customer_date)) AS avg_delivery_days
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date >= '2017-10-01'
  AND order_delivered_customer_date < '2017-11-01'
SQL 변환:   SELECT AVG(JULIANDAY(order_delivered_customer_date) - JULIANDAY(order_purchase_timestamp)) AS avg_delivery_days
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date >= '2017-10-01'
  AND order_delivered_customer_date < '2017-11-01'
Status:     success


   avg_delivery_days
0          11.7

In [59]:
#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement)

with open('Olist_orders_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

✅ 3개 변환 완료!


## 비교

In [119]:
for result in total_result:
  print(result.get('Question'))

2018년 1월에 구매된 주문은 총 몇 건인가요?
2017년 10월에 고객에게 배송 완료된 주문의 평균 배송 소요 시간(구매 시점부터 고객 수령까지, 일 단위)은 얼마인가요?
2018년 1분기(1~3월)에 배송 완료된 주문 중 예상일보다 늦게 도착한 비율은 몇 퍼센트인가요?
2018년 5월에 구매된 주문들의 결제 승인까지 평균 소요 시간(분 단위)은 얼마인가요?
고객 ID 'da973a9c1fa85eb83771f202ed67fc88'의 최신 주문 ID와 상태, 구매 시각을 알려주세요.
2017년 12월 24일에 구매된 주문들의 주문 ID와 상태를 모두 보여주세요.
2018년 2분기(4~6월) 구매된 주문을 상태별로 집계하면 각각 몇 건인가요?
배송사 인계 일자 없이 고객에게 배송 완료된 주문은 몇 건인가요?
2017년에 배송 완료된 주문들의 실제 배송일이 예상 배송일 대비 평균적으로 몇 일 차이 나나요? (실제-예상, 음수는 조기 배송)
2018년 6월에 예상 배송일보다 3일 이상 늦게 도착한 주문의 주문 ID와 구매일, 실제 배송일을 조회해주세요.
2017년에 구매된 주문은 총 몇 건인가요?
2018년 4월에 구매된 주문을 상태별로 집계하면 각각 몇 건인가요?
각 고객별 첫 구매일을 알려주세요.
실제 고객에게 배송된 주문의 평균 배송 소요 시간(구매 시점부터 고객 수령까지, 일 단위)은 얼마인가요?
결제 승인 없이 배송사에 인계된 주문은 몇 건인가요?
2017년 11월에 구매된 주문들의 구매 시점부터 배송사 인계까지 평균 소요 시간(시간 단위)은 얼마인가요?
2018년 3분기(7~9월)에 구매된 주문 중 아직 고객에게 배송되지 않은 비율은 몇 퍼센트인가요?
예상 배송일 대비 실제 배송일의 차이가 가장 큰 상위 10개 주문의 주문 ID와 차이 일수를 알려주세요. (실제-예상, 음수는 조기 배송)
배송 예정일을 지키지 못하고 늦게 도착한 주문의 비율은 얼마인가요?
2017년 5월 10일에 구매된 주문들의 주문 ID와 결제 승인 시각을 모두 보여주세요.


#2.JOIN 있는 SQL문 생성

## df_orders & df_items

DDL문을 다음과 같이 수정. Human Prompt에 DF으로부터 값 예시 넣는 로직 삭제하고 DDL 문의 VALUES로 대체하기

```
DB에 대한 DDL문을 다음 예시처럼 작성해줘.

#테이블명
orders, order_items

#DB orders
order_id	customer_id	order_status	order_purchase_timestamp	order_approved_at	order_delivered_carrier_date	order_delivered_customer_date	order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7	9ef432eb6251297304e76186b10a928d	delivered	2017-10-02 10:56:33	2017-10-02 11:07:15	2017-10-04 19:55:00	2017-10-10 21:25:13	2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451	b0830fb4747a6c6d20dea0b8c802d7ef	delivered	2018-07-24 20:41:37	2018-07-26 03:24:27	2018-07-26 14:31:00	2018-08-07 15:27:45	2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d	41ce2a54c0b03bf3443c3d931a367089	delivered	2018-08-08 08:38:49	2018-08-08 08:55:23	2018-08-08 13:50:00	2018-08-17 18:06:29	2018-09-04 00:00:00

#DB order_items
order_id	order_item_id	product_id	seller_id	shipping_limit_date	price	freight_value
00010242fe8c5a6d1ba2dd792cb16214	1	4244733e06e7ecb4970a6e2683c13e61	48436dade18ac8b2bce089ec2a041202	2017-09-19 09:45:35	58.9	13.29
00018f77f2f0320c557190d7a144bdd3	1	e5f2d52b802189ee658865ca93d83a8f	dd7ddc04e1b6c2c614352b383efe2d36	2017-05-03 11:05:13	239.9	19.93
000229ec398224ef6ca0657da4fc703e	1	c777355d18b72b67abbeef9df44fd0fd	5b51032eddd242adc84c38acab88f23d	2018-01-18 14:48:30	199.0	17.87


#예시
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');

CREATE TABLE farmers_india (id INT, name VARCHAR(255), district_id INT, age INT, income INT); INSERT INTO farmers_india (id, name, district_id, age, income) VALUES (1, 'Farmer A', 1, 45, 50000); CREATE TABLE districts_india (id INT, name VARCHAR(255), state VARCHAR(255)); INSERT INTO districts_india (id, name, state) VALUES (1, 'District A', 'Maharashtra');

CREATE TABLE Armed_Forces (base_id INT, base_name VARCHAR(50), base_location VARCHAR(50), base_type VARCHAR(50)); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (1, 'Fort Bragg', 'North Carolina', 'Army'); INSERT INTO Armed_Forces (base_id, base_name, base_location, base_type) VALUES (2, 'Camp Pendleton', 'California', 'Marines');
```

In [48]:
# 변수 설정
  #1. DDL 선언문
  # 굳이 DDL Statement 2개 따로 쓸 필요가 없음

DDL_Statement = "CREATE TABLE orders (order_id VARCHAR(32) NOT NULL, customer_id VARCHAR(32) NOT NULL, order_status VARCHAR(20) NOT NULL, order_purchase_timestamp DATETIME NOT NULL, order_approved_at DATETIME NULL, order_delivered_carrier_date DATETIME NULL, order_delivered_customer_date DATETIME NULL, order_estimated_delivery_date DATETIME NOT NULL, PRIMARY KEY (order_id)); INSERT INTO orders (order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date) VALUES ('e481f51cbdc54678b7cc49136f2d6af7', '9ef432eb6251297304e76186b10a928d', 'delivered', '2017-10-02 10:56:33', '2017-10-02 11:07:15', '2017-10-04 19:55:00', '2017-10-10 21:25:13', '2017-10-18 00:00:00'), ('53cdb2fc8bc7dce0b6741e2150273451', 'b0830fb4747a6c6d20dea0b8c802d7ef', 'delivered', '2018-07-24 20:41:37', '2018-07-26 03:24:27', '2018-07-26 14:31:00', '2018-08-07 15:27:45', '2018-08-13 00:00:00'), ('47770eb9100c2d0c44946d9cf07ec65d', '41ce2a54c0b03bf3443c3d931a367089', 'delivered', '2018-08-08 08:38:49', '2018-08-08 08:55:23', '2018-08-08 13:50:00', '2018-08-17 18:06:29', '2018-09-04 00:00:00'); CREATE TABLE order_items (order_id CHAR(32) NOT NULL, order_item_id INT NOT NULL, product_id CHAR(32) NOT NULL, seller_id CHAR(32) NOT NULL, shipping_limit_date DATETIME NOT NULL, price DECIMAL(10,2) NOT NULL, freight_value DECIMAL(10,2) NOT NULL, PRIMARY KEY (order_id, order_item_id), FOREIGN KEY (order_id) REFERENCES orders(order_id)); INSERT INTO order_items (order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value) VALUES ('00010242fe8c5a6d1ba2dd792cb16214', 1, '4244733e06e7ecb4970a6e2683c13e61', '48436dade18ac8b2bce089ec2a041202', '2017-09-19 09:45:35', 58.90, 13.29), ('00018f77f2f0320c557190d7a144bdd3', 1, 'e5f2d52b802189ee658865ca93d83a8f', 'dd7ddc04e1b6c2c614352b383efe2d36', '2017-05-03 11:05:13', 239.90, 19.93), ('000229ec398224ef6ca0657da4fc703e', 1, 'c777355d18b72b67abbeef9df44fd0fd', '5b51032eddd242adc84c38acab88f23d', '2018-01-18 14:48:30', 199.00, 17.87);"

  #2. 칼럼 설명 (README 참고)
column_descriptions = """
orders
order_id : unique identifier of the order.
customer_id : key to the customer dataset. Each order has a unique customer_id.
order_status : Reference to the order status (delivered, shipped, etc)
order_purchase_timestamp : Shows the purchase timestamp.
order_approved_at : Shows the payment approval timestamp.
order_delivered_carrier_date : Shows the order posting timestamp. When it was handled to the logistic partner.
order_delivered_customer_date : Shows the actual order delivery date to the customer.
order_estimated_delivery_date : Shows the estimated delivery date that was informed to customer at the purchase moment.

order_items
order_id : order unique identifier
order_item_id : sequential number identifying number of items included in the same order.
product_id : product unique identifier
seller_id : seller unique identifier
shipping_limit_date : Shows the seller shipping limit date for handling the order over to the logistic partner.
price : item price
freight_value : item freight value item (if an order has more than one item the freight value is splitted between items)
"""

#LLM 실행
total_result = []
for i in range(1):
  print(f'진행중 : {i+1}')
  result = generate_sql(DDL_Statement = DDL_Statement,
                        column_descriptions = column_descriptions,
                        history = total_result,
                        is_multi=True)

  #LLM 응답 파싱 후 저장.
  parsed = parse_llm_output(result, 3)
  total_result.extend(parsed)

진행중 : 1
{'input_tokens': 1703, 'output_tokens': 290, 'total_tokens': 1993, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [51]:
print(total_result[0].get('SQL'))

SELECT o.order_id, oi.product_id, oi.price, o.order_status
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE DATE(o.order_purchase_timestamp) = '2018-07-24';


In [ ]:
# 파싱된 SQL 전체 DB에 실행해보기.
conn = sqlite3.connect(':memory:') #원래는 sqlite3.connect('mydb.db')로 Disk에 저장. 지금은 :memory:로 RAM에 저장
df_orders.to_sql('orders', conn, if_exists='replace', index=False)
df_order_items.to_sql('order_items', conn, if_exists='replace', index=False)
execute_res = execute_sql_on_db(total_result, conn)

Question:   2017년 12월에 구매된 주문 중 판매자 ID '94165aea8a35c3c21499cbcae239b16c' 또는 '2c54051840f19eca309a5423cf22df36'의 상품이 포함된 주문의 주문 ID, 주문 상태, 주문별 총 상품 가격 합계와 총 배송비 합계를 알려주세요. 총 상품 가격이 큰 순으로 정렬해 주세요.
SQL 원본:   WITH filt_order_ids AS (
  SELECT o.order_id
  FROM orders o
  WHERE o.order_purchase_timestamp >= '2017-12-01' AND o.order_purchase_timestamp < '2018-01-01'
  INTERSECT
  SELECT oi.order_id
  FROM order_items oi
  WHERE oi.seller_id IN ('94165aea8a35c3c21499cbcae239b16c', '2c54051840f19eca309a5423cf22df36')
)
SELECT o.order_id,
       o.order_status,
       SUM(oi.price) AS total_item_price,
       SUM(oi.freight_value) AS total_freight
FROM filt_order_ids f
JOIN orders o ON o.order_id = f.order_id
JOIN order_items oi ON oi.order_id = f.order_id
GROUP BY o.order_id, o.order_status
ORDER BY total_item_price DESC
Status:     success


   affected_rows
0             -1


--------------------------------------------------
Question:   주문 ID '1764b7f40d0e7f04994494ffabf34ecc'에 새 상품 항목을 추가

In [ ]:
#결과 저장하기.
converted_data = convert_to_gretel_format(total_result, DDL_Statement1, DDL_Statement2)

with open('Olist_orders_n_order_items_text_to_sql_data.json', 'w', encoding='utf-8') as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=4)

print(f"✅ {len(converted_data)}개 변환 완료!")

✅ 3개 변환 완료!


In [ ]:
with open('Olist_orders_n_order_items_text_to_sql_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    # print(json.dumps(data, ensure_ascii=False, indent=4))
print(data[0]['instruction'])

DDL statements:

CREATE TABLE orders (
    order_id                      VARCHAR(32)  NOT NULL,          -- 주문 고유 ID (32자리 hex)
    customer_id                   VARCHAR(32)  NOT NULL,          -- 고객 ID (32자리 hex)
    order_status                  VARCHAR(20)  NOT NULL,          -- 주문 상태 (delivered, shipped, canceled 등)
    order_purchase_timestamp      DATETIME     NOT NULL,          -- 주문 생성 시각
    order_approved_at             DATETIME     NULL,              -- 결제 승인 시각 (미승인 시 NULL)
    order_delivered_carrier_date  DATETIME     NULL,              -- 물류사 인계 시각 (배송 전 NULL)
    order_delivered_customer_date DATETIME     NULL,              -- 고객 수령 시각 (미수령 시 NULL)
    order_estimated_delivery_date DATETIME     NOT NULL,          -- 배송 예정일

    PRIMARY KEY (order_id)
);


CREATE TABLE order_items (
    order_id             CHAR(32)      NOT NULL,          -- 주문 ID (orders.order_id 참조)
    order_item_id        INT           NOT NULL,          -- 주문 내 상품 순번
    product_id           CHAR(3

In [ ]:
print(data[0]['output'])

쿼리 작성: WITH filt_order_ids AS (
  SELECT o.order_id
  FROM orders o
  WHERE o.order_purchase_timestamp >= '2017-12-01' AND o.order_purchase_timestamp < '2018-01-01'
  INTERSECT
  SELECT oi.order_id
  FROM order_items oi
  WHERE oi.seller_id IN ('94165aea8a35c3c21499cbcae239b16c', '2c54051840f19eca309a5423cf22df36')
)
SELECT o.order_id,
       o.order_status,
       SUM(oi.price) AS total_item_price,
       SUM(oi.freight_value) AS total_freight
FROM filt_order_ids f
JOIN orders o ON o.order_id = f.order_id
JOIN order_items oi ON oi.order_id = f.order_id
GROUP BY o.order_id, o.order_status
ORDER BY total_item_price DESC;
